# Simple Machine Learning Baseline

**Goal:** Build a small, reproducible binary-classification baseline and report a trustworthy test accuracy.

Dataset: Wisconsin Diagnostic Breast Cancer dataset, provided by scikit-learn. The dataset contains numeric measurements computed from breast-cancer cell images. The target classes are malignant and benign.

The test set is kept separate from model fitting. The majority-class baseline is included as a simple score to beat.

In [6]:

# Machine Learning Model Comparison
# Compare 3 algorithms on the same dataset and split

import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print("Dataset shape:", X.shape)
print("Classes:", data.target_names)

# 2. Create ONE train/test split for all models
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# 3. Define three different algorithms
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# 4. Train and evaluate each model
results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    # Predictions on the SAME test set
    y_pred = model.predict(X_test)

    # Probability of class 1 for ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

# 5. Display comparison table
results_df = pd.DataFrame(results)

print("\nMODEL COMPARISON")
print(results_df.round(4).to_string(index=False))

# 6. Identify the model with the highest F1 score
best_model = results_df.loc[
    results_df["F1 Score"].idxmax()
]

print("\nBest model by F1 Score:", best_model["Model"])
print("F1 Score:", round(best_model["F1 Score"], 4))


Dataset shape: (569, 30)
Classes: ['malignant' 'benign']
Training samples: 455
Testing samples: 114

MODEL COMPARISON
              Model  Accuracy  Precision  Recall  F1 Score  ROC-AUC
Logistic Regression    0.9825     0.9861  0.9861    0.9861   0.9954
      Decision Tree    0.9123     0.9559  0.9028    0.9286   0.9157
      Random Forest    0.9561     0.9589  0.9722    0.9655   0.9937

Best model by F1 Score: Logistic Regression
F1 Score: 0.9861


In [1]:
# If needed, install dependencies in your environment:
# %pip install scikit-learn pandas

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd


## 1. Load data and split immediately

After loading the dataset, split the features and target right away. The test set is not used to fit the scaler or either model.

In [2]:
dataset = load_breast_cancer()
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Number of features: {X_train.shape[1]}")

Training rows: 455
Test rows: 114
Number of features: 30


## 2. Trivial baseline

A `DummyClassifier` that always predicts the most common class gives a simple reference score. We fit it only on the training set and evaluate it on the test set.

In [3]:
dummy_model = DummyClassifier(strategy="most_frequent")
dummy_model.fit(X_train, y_train)
dummy_predictions = dummy_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, dummy_predictions)

print(f"Trivial baseline test accuracy: {baseline_accuracy:.3f}")

Trivial baseline test accuracy: 0.632


## 3. Fit a simple model

Use logistic regression. Standardization is inside a pipeline, so the scaler learns from the training data only.

In [4]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)

model.fit(X_train, y_train)
test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Logistic regression test accuracy: {test_accuracy:.3f}")

Logistic regression test accuracy: 0.982


## 4. Compare scores

The final reported model score is the accuracy on the held-out test set. Run the notebook to calculate the exact scores in your environment.

In [5]:
results = pd.DataFrame({
    "Model": ["Most-frequent dummy baseline", "Logistic regression"],
    "Test accuracy": [baseline_accuracy, test_accuracy],
})
results

,Model,Test accuracy
0,Most-frequent dummy baseline,0.631579
1,Logistic regression,0.982456


## Short conclusion

- The dummy classifier is the trivial baseline to beat.
- Logistic regression is the first real model.
- The reported accuracy comes from a held-out test set that was not used for fitting.
- This is a learning baseline, not a medical diagnostic tool. Accuracy alone does not describe every type of model error.